# 01 — Data Collection and Inspection

**Goal:** download the official MovieLens Latest Small dataset, create the project folders, load the four CSV files, and inspect their structure. Run this notebook first.

## 1. Create the project structure

The code creates the folders used by every later notebook. Run from the project root or from `02_Notebooks`.

In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "02_Notebooks" else Path.cwd()
folders = [
    "01_Data/Raw", "01_Data/Processed", "02_Notebooks",
    "03_Models", "04_App", "05_Visualizations"
]
for folder in folders:
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)
print("Created project folders under:", PROJECT_ROOT)

Created project folders under: D:\DataScience\notebooks\CineMatch_Ready_Notebooks


## 2. Download MovieLens Latest Small

This uses the official GroupLens ZIP file. An internet connection is required for the first run. The archive contains `movies.csv`, `ratings.csv`, `tags.csv`, and `links.csv`.

In [6]:
# Extract the downloaded MovieLens ZIP file into the CineMatch project.

from pathlib import Path
import zipfile

# Locate the Downloads folder
DOWNLOAD_DIR = Path.home() / "Downloads"

# Locate the project's Raw data folder
RAW_DIR = PROJECT_ROOT / "01_Data" / "Raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Find the downloaded MovieLens ZIP file
zip_files = list(DOWNLOAD_DIR.glob("ml-latest-small*.zip"))

if not zip_files:
    raise FileNotFoundError(
        "MovieLens ZIP file not found in Downloads. "
        "Check that ml-latest-small.zip is there."
    )

# Select the most recently downloaded matching ZIP
zip_path = max(zip_files, key=lambda f: f.stat().st_mtime)

print("ZIP file found:", zip_path)

# Extract the dataset
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(RAW_DIR)

print("\nDataset extracted successfully!")

# Verify the extracted files
DATA_DIR = RAW_DIR / "ml-latest-small"

required_files = ["movies.csv", "ratings.csv", "tags.csv", "links.csv"]

for file in required_files:
    file_path = DATA_DIR / file

    if file_path.exists():
        print("FOUND:", file)
    else:
        print("MISSING:", file)

if all((DATA_DIR / file).exists() for file in required_files):
    print("\nSUCCESS! MovieLens dataset is ready.")
else:
    print("\nDataset extraction needs checking.")

ZIP file found: C:\Users\HP\Downloads\ml-latest-small.zip

Dataset extracted successfully!
FOUND: movies.csv
FOUND: ratings.csv
FOUND: tags.csv
FOUND: links.csv

SUCCESS! MovieLens dataset is ready.


## 3. Load the four datasets

- **movies.csv:** movie ID, title, and genre labels.
- **ratings.csv:** user ID, movie ID, rating (0.5–5.0), and timestamp.
- **tags.csv:** user-supplied descriptive tags for movies.
- **links.csv:** external IMDb and TMDb identifiers.

In [7]:
import pandas as pd

movies = pd.read_csv(extract_dir / "movies.csv")
ratings = pd.read_csv(extract_dir / "ratings.csv")
tags = pd.read_csv(extract_dir / "tags.csv")
links = pd.read_csv(extract_dir / "links.csv")

datasets = {"movies": movies, "ratings": ratings, "tags": tags, "links": links}
for name, frame in datasets.items():
    print(f"{name:8s}: {frame.shape[0]:,} rows × {frame.shape[1]} columns")

movies  : 9,742 rows × 3 columns
ratings : 100,836 rows × 4 columns
tags    : 3,683 rows × 4 columns
links   : 9,742 rows × 3 columns


## 4. Preview and inspect columns and data types

In [8]:
for name, frame in datasets.items():
    print("\\n" + "=" * 70, "\\n", name.upper())
    display(frame.head())
    display(frame.tail(3))
    print("Columns:", frame.columns.tolist())
    print("\\nData types:")
    display(frame.dtypes.to_frame("dtype"))

\n====================================================================== \n MOVIES


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


,movieId,title,genres
9739,193585,Flint (2017),Drama
9740,193587,Bungo Stray Dogs: Dead Apple (2018),Action|Animation
9741,193609,Andrew Dice Clay: Dice Rules (1991),Comedy


Columns: ['movieId', 'title', 'genres']
\nData types:


,dtype
movieId,int64
title,str
genres,str


\n====================================================================== \n RATINGS


... (10 outputs left out)

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


,movieId,imdbId,tmdbId
9739,193585,6397426,479308.0
9740,193587,8391976,483455.0
9741,193609,101726,37891.0


Columns: ['movieId', 'imdbId', 'tmdbId']
\nData types:


,dtype
movieId,int64
imdbId,int64
tmdbId,float64


## 5. Data quality checks

Check missing values, duplicate rows, unique users/movies, genres, and the rating range. These are observations from the downloaded data, not assumed results.

In [9]:
for name, frame in datasets.items():
    print(f"\\n{name.upper()}")
    print("Missing values:", int(frame.isna().sum().sum()))
    print("Duplicate rows:", int(frame.duplicated().sum()))

print("\\nUnique users in ratings:", ratings["userId"].nunique())
print("Unique movies in movies:", movies["movieId"].nunique())
print("Unique movies rated:", ratings["movieId"].nunique())
print("Rating values:", sorted(ratings["rating"].dropna().unique()))
all_genres = sorted({g for cell in movies["genres"].dropna()
                     for g in cell.split("|") if g != "(no genres listed)"})
print("Unique genres:", all_genres)
display(ratings["rating"].value_counts().sort_index().rename("count").to_frame())

\nMOVIES
Missing values: 0
Duplicate rows: 0
\nRATINGS
Missing values: 0
Duplicate rows: 0
\nTAGS
Missing values: 0
Duplicate rows: 0
\nLINKS
Missing values: 8
Duplicate rows: 0
\nUnique users in ratings: 610
Unique movies in movies: 9742
Unique movies rated: 9724
Rating values: [np.float64(0.5), np.float64(1.0), np.float64(1.5), np.float64(2.0), np.float64(2.5), np.float64(3.0), np.float64(3.5), np.float64(4.0), np.float64(4.5), np.float64(5.0)]
Unique genres: ['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


,count
rating,
0.5,1370
1.0,2811
1.5,1791
2.0,7551
2.5,5550
3.0,20047
3.5,13136
4.0,26818
4.5,8551


## 6. Check relationships between files

`movieId` links movies to ratings, tags, and links. `userId` links ratings to tags. The following checks reveal records that cannot be matched to the movie catalogue.

In [10]:
movie_ids = set(movies["movieId"])
for name, frame in [("ratings", ratings), ("tags", tags), ("links", links)]:
    unmatched = (~frame["movieId"].isin(movie_ids)).sum()
    print(f"{name}: {unmatched:,} rows have movieId absent from movies.csv")

print("Users in ratings:", ratings["userId"].nunique())
print("Users in tags:", tags["userId"].nunique())

ratings: 0 rows have movieId absent from movies.csv
tags: 0 rows have movieId absent from movies.csv
links: 0 rows have movieId absent from movies.csv
Users in ratings: 610
Users in tags: 58


## 7. Save a compact inspection report

The report records dimensions and quality checks so the project has a reproducible data-inspection output.

In [11]:
report = []
for name, frame in datasets.items():
    report.append({
        "dataset": name, "rows": len(frame), "columns": len(frame.columns),
        "missing_cells": int(frame.isna().sum().sum()),
        "duplicate_rows": int(frame.duplicated().sum())
    })
inspection_report = pd.DataFrame(report)
display(inspection_report)
inspection_report.to_csv(PROJECT_ROOT / "01_Data" / "Raw" / "inspection_report.csv", index=False)
print("Saved inspection_report.csv")

,dataset,rows,columns,missing_cells,duplicate_rows
0,movies,9742,3,0,0
1,ratings,100836,4,0,0
2,tags,3683,4,0,0
3,links,9742,3,8,0


Saved inspection_report.csv


## Completion check

You should now have the four original CSV files inside `01_Data/Raw/ml-latest-small/` and `inspection_report.csv` in `01_Data/Raw/`. Next: **02_Data_Cleaning_and_Preprocessing.ipynb**.